# Train Small Audio Encoder for ESP32

Train a tiny mel-spectrogram → 256-dim model (~100-300KB) for on-device inference.

## Architecture
- Input: Mel-spectrogram (64 bins × 96 frames)
- Audio Encoder: Small CNN → 256-dim embedding
- Text Projection: MPNet 768 → 256-dim
- Training: Contrastive loss aligns audio and text

## Output
- `audio_encoder.tflite` - Small model for ESP32 (~100-300KB)
- `embeddings.bin` - Pre-computed text embeddings (256-dim)
- `intents.txt` - Question strings

In [14]:
from pathlib import Path

# Paths
WORK_DIR = Path('/workspace') if Path('/workspace').exists() else Path('.').resolve()
AUDIO_DIR = WORK_DIR / 'audio_data'
MODEL_DIR = WORK_DIR / 'models'
DATA_DIR = WORK_DIR / 'data' / 'raw'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Dimensions
MEL_BINS = 64
MEL_FRAMES = 96
EMBEDDING_DIM = 256
TEXT_DIM = 768

# Audio
SAMPLE_RATE = 16000
HOP_LENGTH = 160
N_FFT = 512

# Training
BATCH_SIZE = 32
EPOCHS = 3000
LEARNING_RATE = 1e-4
TEMPERATURE = 0.2

print(f"Work dir: {WORK_DIR}")
print(f"Audio encoder: ({MEL_BINS}×{MEL_FRAMES}) → {EMBEDDING_DIM}-dim")

Work dir: /workspace
Audio encoder: (64×96) → 256-dim


In [15]:
# Install dependencies
!uv add tensorflow librosa soundfile sentence-transformers pandas numpy matplotlib tqdm scikit-learn
print("Done!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Resolved 174 packages in 8ms
Audited 166 packages in 8ms
Done!


In [16]:
import tensorflow as tf
import numpy as np
import pandas as pd
import librosa
import json
import struct
from tqdm import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow: {tf.__version__}")

TensorFlow: 2.20.0


## Load Metadata

In [17]:
# Load metadata from all workers
metadata = []
for mf in sorted(AUDIO_DIR.glob('metadata_worker_*.json')):
    with open(mf) as f:
        metadata.extend(json.load(f))

# Filter to existing files
metadata = [m for m in tqdm(metadata, desc="Checking files") if Path(m['file']).exists()]
print(f"Total samples: {len(metadata)}")

# Load questions
qa_file = DATA_DIR / 'full.csv'
if qa_file.exists():
    questions = pd.read_csv(qa_file)['question'].tolist()
    print(f"Questions: {len(questions)}")
else:
    questions = list(dict.fromkeys([m['text'] for m in metadata]))
    print(f"Questions (from metadata): {len(questions)}")

Checking files: 100%|██████████| 31504/31504 [00:00<00:00, 67873.26it/s]

Total samples: 31504
Questions: 2008


## Extract Mel-Spectrograms

In [18]:
def compute_mel(audio_path):
    """Compute mel-spectrogram (64×96) from audio file."""
    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)
    
    # Pad if too short
    min_len = HOP_LENGTH * (MEL_FRAMES - 1) + N_FFT
    if len(audio) < min_len:
        audio = np.pad(audio, (0, min_len - len(audio)))
    
    # Compute mel-spectrogram
    mel = librosa.feature.melspectrogram(
        y=audio, sr=SAMPLE_RATE, n_mels=MEL_BINS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, fmin=125.0, fmax=7500.0
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    # Truncate/pad to 96 frames
    if mel_db.shape[1] < MEL_FRAMES:
        mel_db = np.pad(mel_db, ((0,0), (0, MEL_FRAMES - mel_db.shape[1])))
    else:
        mel_db = mel_db[:, :MEL_FRAMES]
    
    # Normalize to [0, 1]
    mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
    return mel_db.astype(np.float32)

# Test
test_mel = compute_mel(metadata[0]['file'])
print(f"Mel shape: {test_mel.shape}")

Mel shape: (64, 96)


In [19]:
# Extract all mel-spectrograms
print(f"Extracting mel-spectrograms for {len(metadata)} files...")

mel_specs = []
valid_metadata = []

for item in tqdm(metadata):
    try:
        mel = compute_mel(item['file'])
        mel_specs.append(mel)
        valid_metadata.append(item)
    except Exception as e:
        continue

mel_specs = np.array(mel_specs)
print(f"Mel-spectrograms: {mel_specs.shape}")

Extracting mel-spectrograms for 31504 files...


100%|██████████| 31504/31504 [35:03<00:00, 14.98it/s]  


Mel-spectrograms: (31504, 64, 96)


## Get Text Embeddings

In [20]:
!uv add tf-keras

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Resolved 174 packages in 3ms
Audited 166 packages in 6ms


In [21]:
from sentence_transformers import SentenceTransformer

print("Loading text encoder...")
text_encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Get unique texts and create mapping
unique_texts = list(dict.fromkeys([m['text'] for m in valid_metadata]))
text_to_emb = {t: text_encoder.encode(t) for t in tqdm(unique_texts, desc="Encoding")}

# Map to each sample
text_embeddings = np.array([text_to_emb[m['text']] for m in valid_metadata])
bucket_ids = np.array([m['bucket_id'] for m in valid_metadata])

print(f"Text embeddings: {text_embeddings.shape}")

Loading text encoder...


Encoding: 100%|██████████| 1948/1948 [00:33<00:00, 58.36it/s]


Text embeddings: (31504, 768)


## Build Audio Encoder (Small CNN)

In [22]:
def build_audio_encoder():
    """Small CNN: (64, 96) mel → 256-dim embedding. Target: ~100-300KB TFLite."""
    inputs = tf.keras.Input(shape=(MEL_BINS, MEL_FRAMES, 1))
    
    # Conv blocks with aggressive pooling
    x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    x = tf.keras.layers.MaxPooling2D(2)(x)  # 32×48
    x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)  # 16×24
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.MaxPooling2D(2)(x)  # 8×12
    x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)  # 128
    
    # Project to embedding
    x = tf.keras.layers.Dense(EMBEDDING_DIM)(x)
    outputs = tf.keras.layers.Lambda(lambda x: tf.nn.l2_normalize(x, axis=1))(x)
    
    return tf.keras.Model(inputs, outputs, name='audio_encoder')

def build_text_projection():
    """Project MPNet 768 → 256."""
    inputs = tf.keras.Input(shape=(TEXT_DIM,))
    x = tf.keras.layers.Dense(512, activation='relu')(inputs)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Dense(EMBEDDING_DIM)(x)
    outputs = tf.keras.layers.Lambda(lambda x: tf.nn.l2_normalize(x, axis=1))(x)
    return tf.keras.Model(inputs, outputs, name='text_projection')

audio_encoder = build_audio_encoder()
text_projection = build_text_projection()

print("Audio Encoder:")
audio_encoder.summary()
print(f"\nText Projection: {TEXT_DIM} → {EMBEDDING_DIM}")

Audio Encoder:
Model: "audio_encoder"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 64, 96, 1)]       0         
                                                                 
 conv2d_4 (Conv2D)           (None, 64, 96, 32)        320       
                                                                 
 max_pooling2d_3 (MaxPoolin  (None, 32, 48, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_5 (Conv2D)           (None, 32, 48, 64)        18496     
                                                                 
 max_pooling2d_4 (MaxPoolin  (None, 16, 24, 64)        0         
 g2D)                                                            
                                                                 
 conv2d_6 (Conv2D)           (None, 16

## Training

In [11]:
# Batch generator (one sample per bucket)
class BucketBatchGen:
    def __init__(self, mels, texts, buckets, batch_size):
        self.mels = mels
        self.texts = texts
        self.bucket_to_idx = defaultdict(list)
        for i, b in enumerate(buckets):
            self.bucket_to_idx[b].append(i)
        self.all_buckets = list(self.bucket_to_idx.keys())
        self.batch_size = min(batch_size, len(self.all_buckets))
    
    def __len__(self):
        return len(self.mels) // self.batch_size
    
    def generate(self):
        import random
        buckets = random.sample(self.all_buckets, self.batch_size)
        mel_batch, text_batch = [], []
        for b in buckets:
            idx = random.choice(self.bucket_to_idx[b])
            mel_batch.append(self.mels[idx])
            text_batch.append(self.texts[idx])
        return np.array(mel_batch), np.array(text_batch)

batch_gen = BucketBatchGen(mel_specs, text_embeddings, bucket_ids, BATCH_SIZE)
print(f"Batches per epoch: {len(batch_gen)}")

Batches per epoch: 984


In [12]:
# Contrastive loss and training step
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)
all_vars = audio_encoder.trainable_variables + text_projection.trainable_variables

@tf.function
def train_step(mel_batch, text_batch):
    with tf.GradientTape() as tape:
        # Add channel dim for CNN
        mel_input = tf.expand_dims(mel_batch, -1)
        
        audio_emb = audio_encoder(mel_input, training=True)
        text_emb = text_projection(text_batch, training=True)
        
        # Contrastive loss
        logits = tf.matmul(audio_emb, text_emb, transpose_b=True) / TEMPERATURE
        labels = tf.range(tf.shape(audio_emb)[0])
        loss_a2t = tf.nn.sparse_softmax_cross_entropy_with_logits(labels, logits)
        loss_t2a = tf.nn.sparse_softmax_cross_entropy_with_logits(labels, tf.transpose(logits))
        loss = (tf.reduce_mean(loss_a2t) + tf.reduce_mean(loss_t2a)) / 2
    
    grads = tape.gradient(loss, all_vars)
    optimizer.apply_gradients(zip(grads, all_vars))
    return loss

print("Training setup done")

Training setup done


In [ ]:
# Training loop
print(f"Training for {EPOCHS} epochs...")
history = []

for epoch in range(1000):
    losses = []
    for _ in range(len(batch_gen)):
        mel_batch, text_batch = batch_gen.generate()
        loss = train_step(
            tf.constant(mel_batch, dtype=tf.float32),
            tf.constant(text_batch, dtype=tf.float32)
        )
        losses.append(loss.numpy())
    
    avg_loss = np.mean(losses)
    history.append(avg_loss)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f}")

print(f"\nFinal loss: {history[-1]:.4f}")

Training for 3000 epochs...


## Evaluation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Get embeddings
mel_input = np.expand_dims(mel_specs, -1)
audio_embs = audio_encoder.predict(mel_input, verbose=0)
text_embs = text_projection.predict(text_embeddings, verbose=0)

# Bucket accuracy
sample_size = min(500, len(audio_embs))
sim = cosine_similarity(audio_embs[:sample_size], text_embs[:sample_size])
preds = np.argmax(sim, axis=1)
pred_buckets = bucket_ids[:sample_size][preds]
true_buckets = bucket_ids[:sample_size]
acc = np.mean(pred_buckets == true_buckets)

print(f"Bucket Accuracy: {acc*100:.1f}%")
print(f"Mean diagonal sim: {np.mean(np.diag(sim)):.3f}")

## Export for ESP32

In [ ]:
# Save audio encoder as TFLite (int8 quantized)
converter = tf.lite.TFLiteConverter.from_keras_model(audio_encoder)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.int8]

# Representative dataset for quantization
def representative_data():
    for i in range(min(100, len(mel_specs))):
        yield [np.expand_dims(mel_specs[i:i+1], -1).astype(np.float32)]

converter.representative_dataset = representative_data
tflite_model = converter.convert()

tflite_path = MODEL_DIR / 'audio_encoder.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Audio encoder TFLite: {tflite_path}")
print(f"Size: {len(tflite_model) / 1024:.1f} KB")

In [ ]:
# Generate and save text embeddings
query_text_embs = text_encoder.encode(questions)
query_embs = text_projection.predict(query_text_embs, verbose=0)

# Save embeddings.bin
emb_path = MODEL_DIR / 'embeddings.bin'
with open(emb_path, 'wb') as f:
    for emb in query_embs:
        f.write(struct.pack(f'{EMBEDDING_DIM}f', *emb))

print(f"Embeddings: {emb_path} ({len(query_embs)} × {EMBEDDING_DIM})")
print(f"Size: {emb_path.stat().st_size / 1024:.1f} KB")

# Save intents.txt
int_path = MODEL_DIR / 'intents.txt'
with open(int_path, 'w') as f:
    for q in questions:
        f.write(f"{q}\n")

print(f"Intents: {int_path} ({len(questions)} questions)")

In [ ]:
print("="*60)
print("TRAINING COMPLETE")
print("="*60)
print(f"\nOutput files:")
for f in sorted(MODEL_DIR.glob('*')):
    if f.is_file():
        print(f"  {f.name}: {f.stat().st_size/1024:.1f} KB")

print(f"\nFor ESP32:")
print(f"  audio_encoder.tflite → SD:/models/audio_encoder.tflite")
print(f"  embeddings.bin → SD:/data/embeddings.bin")
print(f"  intents.txt → SD:/data/intents.txt")